# Demonstrasi: Statistik Deskriptif & Interpretasi Awal — PIF304 Pengantar Data Sains
**Pertemuan 4 — Sub-CPMK S02: Menganalisis data secara deskriptif**

Pekan lalu kita menyusun *data dictionary*. Hari ini kamus itu dipakai: **skala ukur menentukan ringkasan yang sah**.

Alur demo:
1. Skala ukur → pilihan statistik
2. Ukuran pemusatan dan penyebaran
3. Bentuk distribusi (histogram, boxplot)
4. Data kategorikal: frekuensi, proporsi, tabulasi silang
5. Korelasi dan batasnya
6. Sampling: mengapa 45 responden belum tentu mewakili
7. Menulis interpretasi yang jujur

**Dataset:** survei 45 mahasiswa (nama sudah dianonimkan).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
# Palet grafik (sudah diuji aman untuk buta warna): satu seri = TEAL; pembanding = ORANGE, VIOLET; latar = GREY
NAVY, TEAL, ORANGE, VIOLET, GREY = "#13233F", "#1293A6", "#E4683A", "#6C63C9", "#B8BDC4"
PALET = [TEAL, ORANGE, VIOLET]

### Unggah File Data
Jalankan sel di bawah, klik **Choose Files**, lalu pilih **`Data_Survei_Mahasiswa_Anon_PDS.xlsx`** yang sudah Anda unduh dari LMS.

In [ ]:
try:
    from google.colab import files
    print("Silakan pilih file: Data_Survei_Mahasiswa_Anon_PDS.xlsx")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih file-nya.")
    nama_file = list(uploaded.keys())[0]
except ImportError:
    # Di luar Google Colab (mis. Jupyter lokal): letakkan file di folder yang sama dengan notebook
    nama_file = "Data_Survei_Mahasiswa_Anon_PDS.xlsx"

df = pd.read_excel(nama_file)   # sheet pertama = "Data"
print("File:", nama_file, "| ukuran:", df.shape)
df.head()

## 1. Skala Ukur Menentukan Statistik
Kita tuliskan ulang skala ukur dari data dictionary (Pertemuan 3), lalu memakainya sebagai panduan.

In [ ]:
skala = {
    "usia": "rasio", "jarak_km": "rasio", "waktu_tempuh_menit": "rasio",
    "tingkat_ketertarikan": "ordinal", "frekuensi_pakai_ai": "ordinal", "penggunaan_internet": "ordinal", "semester": "ordinal",
    "perangkat_utama": "nominal", "moda_transportasi": "nominal", "platform_belajar": "nominal", "waktu_belajar_favorit": "nominal",
}
statistik_sah = {
    "nominal": "frekuensi, proporsi, modus",
    "ordinal": "frekuensi, proporsi, modus, median, kuartil",
    "rasio":   "semua di atas + mean, simpangan baku, rentang",
}
pd.DataFrame({"skala": skala}).assign(statistik_yang_sah=lambda t: t["skala"].map(statistik_sah))

## 2. Ukuran Pemusatan: Mean, Median, Modus
Mulai dari kolom yang "sehat": `waktu_tempuh_menit`.

In [ ]:
w = df["waktu_tempuh_menit"]
print(f"Mean   : {w.mean():.2f} menit")
print(f"Median : {w.median():.1f} menit")
print(f"Modus  : {w.mode().tolist()} menit")

Mean sedikit lebih besar daripada median → ada beberapa responden dengan waktu tempuh panjang yang "menarik" rata-rata ke kanan.

Sekarang kolom yang bermasalah: `jarak_km` (ingat empat isian 500/1000).

In [ ]:
j = df["jarak_km"]
wajar = j < 100
print(f"Mean (semua data)         : {j.mean():.2f} km")
print(f"Median (semua data)       : {j.median():.2f} km")
print(f"Mean (tanpa 4 nilai aneh) : {j[wajar].mean():.2f} km")
print(f"Median (tanpa 4 nilai aneh): {j[wajar].median():.2f} km")

**Pelajaran:** empat nilai mengubah mean dari sekitar 2 km menjadi 57 km, sedangkan median nyaris tidak bergerak. Median **tahan (robust)** terhadap nilai ekstrem; mean tidak.

> Aturan praktis: untuk data yang miring atau memiliki nilai ekstrem, laporkan **median** (dan IQR). Untuk data yang kira-kira simetris, mean (dan simpangan baku) boleh dipakai. Paling aman: laporkan keduanya.

## 3. Ukuran Penyebaran
Dua kelompok bisa memiliki rata-rata yang sama tetapi sebaran yang sangat berbeda. Rata-rata tanpa sebaran adalah ringkasan yang setengah jujur.

In [ ]:
def ringkas(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return pd.Series({
        "n": s.count(), "min": s.min(), "Q1": q1, "median": s.median(), "Q3": q3, "max": s.max(),
        "rentang": s.max() - s.min(), "IQR": q3 - q1, "mean": round(s.mean(), 2), "simpangan_baku": round(s.std(), 2),
    })

pd.DataFrame({"usia": ringkas(df["usia"]), "waktu_tempuh_menit": ringkas(df["waktu_tempuh_menit"]), "jarak_km": ringkas(df["jarak_km"])})

Perhatikan `jarak_km`: simpangan baku jauh lebih besar daripada IQR. Itu tanda bahwa sebaran "dikuasai" oleh segelintir nilai ekstrem.

`df.describe()` memberi ringkasan yang sama dengan cepat — tetapi ia **tidak tahu skala ukur**. Ia akan menghitung mean untuk `tingkat_ketertarikan` (ordinal) tanpa protes.

In [ ]:
df.describe().round(2)

## 4. Bentuk Distribusi
Angka ringkasan menyembunyikan bentuk. Selalu **lihat** datanya.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.histplot(df["waktu_tempuh_menit"], bins=range(0, 30, 3), color=TEAL, ax=axes[0])
axes[0].axvline(w.mean(), color=ORANGE, ls="--", label=f"mean = {w.mean():.1f}")
axes[0].axvline(w.median(), color=NAVY, ls="-", label=f"median = {w.median():.0f}")
axes[0].set(title="Histogram waktu tempuh", xlabel="menit", ylabel="jumlah responden"); axes[0].legend()

sns.boxplot(x=df["waktu_tempuh_menit"], color=TEAL, ax=axes[1])
axes[1].set(title="Boxplot waktu tempuh (ringkasan lima angka)", xlabel="menit")
plt.tight_layout(); plt.show()

print("Kemiringan (skewness):", round(w.skew(), 2), "-> positif = ekor panjang ke kanan")

**Membaca boxplot:** kotak = Q1 sampai Q3 (50% data di tengah), garis di dalam = median, "kumis" = sampai 1,5 × IQR, titik = nilai di luar itu. Titik di luar kumis **bukan otomatis salah** — ia hanya "berbeda dari kebanyakan" dan perlu diperiksa.

## 5. Data Kategorikal
Untuk skala nominal dan ordinal, ringkasan utamanya adalah **frekuensi dan proporsi**.

In [ ]:
def tabel_frekuensi(s):
    t = s.value_counts().to_frame("jumlah")
    t["proporsi"] = (t["jumlah"] / t["jumlah"].sum()).round(3)
    return t

tabel_frekuensi(df["platform_belajar"])

Untuk skala **ordinal**, urutan kategori bermakna. Kita beri tahu pandas urutannya, lalu median dapat dihitung.

In [ ]:
urut_ai = ["Kadang-kadang", "Sering", "Sangat sering"]
ai = pd.Categorical(df["frekuensi_pakai_ai"], categories=urut_ai, ordered=True)
kode = pd.Series(ai.codes)          # 0, 1, 2
print("Frekuensi:")
print(pd.Series(ai).value_counts().reindex(urut_ai).to_string())
print("\nKategori median:", urut_ai[int(kode.median())])

print("\ntingkat_ketertarikan -> median =", df["tingkat_ketertarikan"].median(), "| modus =", df["tingkat_ketertarikan"].mode().tolist())
print("(mean =", round(df["tingkat_ketertarikan"].mean(), 2), "sering dilaporkan untuk skala Likert, tetapi mengandaikan jarak antarskor sama)")

### Tabulasi silang (dua variabel kategorikal)
Apakah frekuensi pemakaian AI berbeda antara Semester 3 dan Semester 5?

In [ ]:
jumlah = pd.crosstab(df["semester"], pd.Series(ai, name="frekuensi_pakai_ai"))
persen = pd.crosstab(df["semester"], pd.Series(ai, name="frekuensi_pakai_ai"), normalize="index").mul(100).round(1)
print("Jumlah responden:"); print(jumlah, "\n")
print("Persentase per baris (%):"); print(persen)

**Interpretasi yang jujur:** pada sampel ini, proporsi "Sangat sering" lebih tinggi di Semester 5 dibanding Semester 3. Selalu laporkan **persentase bersama jumlah mentahnya** — "48%" dari 21 orang hanyalah 10 orang.

## 6. Korelasi dan Batasnya
Apakah jarak berkaitan dengan waktu tempuh? Kita bandingkan korelasi **dengan** dan **tanpa** empat nilai jarak yang mencurigakan.

In [ ]:
def korelasi(d, label):
    r_p = d["jarak_km"].corr(d["waktu_tempuh_menit"], method="pearson")
    r_s = d["jarak_km"].corr(d["waktu_tempuh_menit"], method="spearman")
    print(f"{label:<28} n={len(d):>2} | Pearson r = {r_p:+.2f} | Spearman rho = {r_s:+.2f}")

korelasi(df, "Semua data")
korelasi(df[wajar], "Tanpa 4 nilai mencurigakan")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.scatterplot(data=df, x="jarak_km", y="waktu_tempuh_menit", color=ORANGE, ax=axes[0])
axes[0].set(title="Semua data (4 titik ekstrem mendominasi)")
sns.scatterplot(data=df[wajar], x="jarak_km", y="waktu_tempuh_menit", hue="moda_transportasi", palette=PALET, ax=axes[1])
axes[1].set(title="Tanpa 4 nilai mencurigakan")
plt.tight_layout(); plt.show()

**Pelajaran:**
- Pearson sangat peka terhadap nilai ekstrem; Spearman (berbasis peringkat) lebih tahan.
- Korelasi mengukur **kekuatan hubungan linear/monoton**, bukan sebab-akibat. Jarak dan waktu tempuh memang masuk akal berhubungan, tetapi **moda transportasi** ikut berperan (variabel ketiga).
- Selalu gambar *scatter plot* sebelum memercayai satu angka korelasi.

## 7. Sampling: Seberapa Stabil Angka Kita?
29 dari 45 responden (64%) memilih belajar **malam**. Andaikan 45 responden ini adalah "populasi kecil", lalu kita hanya sempat menyurvei **15 orang** secara acak. Berapa proporsi yang akan kita peroleh? Kita ulangi 1.000 kali.

In [ ]:
rng = np.random.default_rng(4)
malam = (df["waktu_belajar_favorit"] == "Malam").to_numpy()
simulasi = [rng.choice(malam, size=15, replace=False).mean() for _ in range(1000)]

fig, ax = plt.subplots(figsize=(7, 3.4))
sns.histplot(simulasi, bins=np.arange(0.2, 1.05, 1/15), color=TEAL, ax=ax)
ax.axvline(malam.mean(), color=ORANGE, ls="--", label=f"proporsi 45 responden = {malam.mean():.0%}")
ax.set(title="Proporsi 'Malam' pada 1.000 sampel acak berukuran 15", xlabel="proporsi sampel", ylabel="jumlah simulasi"); ax.legend()
plt.tight_layout(); plt.show()

print(f"Rentang hasil 90% simulasi: {np.quantile(simulasi, 0.05):.0%} s.d. {np.quantile(simulasi, 0.95):.0%}")

**Pelajaran:** sampel yang berbeda memberi angka yang berbeda (*variabilitas sampling*). Makin kecil sampel, makin lebar goyangannya.

Dan ini baru sampling **acak**. Survei kita **bukan** sampel acak: yang mengisi adalah yang bersedia mengisi (*self-selection*). Ketidakpastian yang sebenarnya lebih besar dan arahnya tidak diketahui. Karena itu, kalimat kita harus selalu dibatasi pada "**responden survei ini**".

## 8. Menulis Interpretasi Awal
Pola kalimat yang aman: **[siapa/berapa] + [ukuran yang tepat] + [sebaran] + [batas]**.

In [ ]:
print(f"""Contoh interpretasi:
Pada {len(df)} responden survei, median waktu tempuh ke kampus adalah {w.median():.0f} menit
(IQR {w.quantile(.25):.0f}-{w.quantile(.75):.0f} menit; rentang {w.min()}-{w.max()} menit).
Sebanyak {malam.sum()} responden ({malam.mean():.0%}) memilih malam sebagai waktu belajar favorit.
Angka ini menggambarkan responden yang mengisi survei dan belum tentu mewakili seluruh mahasiswa.""")

## Ringkasan
| Pertanyaan | Alat | Catatan kejujuran |
|---|---|---|
| Di mana pusat data? | mean / median / modus | Sesuaikan dengan skala ukur dan bentuk distribusi |
| Seberapa menyebar? | rentang, IQR, simpangan baku | Rata-rata tanpa sebaran belum lengkap |
| Bentuknya bagaimana? | histogram, boxplot | Lihat dulu, hitung kemudian |
| Kategori mana yang dominan? | frekuensi, proporsi, crosstab | Sertakan jumlah mentah |
| Apakah dua variabel berhubungan? | scatter, Pearson/Spearman | Korelasi ≠ kausalitas |
| Dapatkah digeneralisasi? | pikirkan cara sampel diambil | Batasi klaim pada responden |

**Berikutnya (Pertemuan 5):** apa yang harus dilakukan terhadap nilai `jarak_km` yang aneh, data ganda, dan nilai kosong — dan bagaimana mencatat keputusannya.